<a href="https://colab.research.google.com/github/cdxing/applied-scientist-training/blob/main/01_prime_retention/Prime_Membership_Retention_Applied_Scientist_Mini_Lab_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# START HERE TOMORROW

### 5-minute retrieval before looking at notes
1. Why is long-term personalization difficult?
2. What tradeoff is Progressive Horizon Learning trying to solve?
3. How would I test whether the proposed method actually helps?

### Hands-on task
Train Logistic Regression baseline:
features → logit → sigmoid → probability → prediction

Then inspect:
- class balance
- precision / recall / ROC-AUC
- coefficient signs
- failure cases

Business problem:
Predict which current members are at higher risk of cancelling their membership.

ML formulation:
Binary classification.

Target:
churned = 1 if the member cancels, otherwise 0.

Possible features:
- membership tenure
- monthly purchase frequency
- Prime Video engagement
- delivery usage
- average order value
- customer service contacts
- recent change in activity

Goal:
Use the model to identify higher-risk members and support targeted retention interventions.

## 1. Imports and Reproducibility

**What**  
Import numerical, tabular-data, and train/test-splitting tools.

**Why**  
We need a minimal toolkit for generating data, manipulating features, and creating reproducible dataset splits.

**How**  
Use NumPy for numerical operations, pandas for tabular data, and scikit-learn for ML utilities.

**Uniqueness / What to notice**  
`SEED = 42` makes random data generation and splitting reproducible, which is important for debugging and fair model comparison.

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

SEED = 42
rng = np.random.default_rng(SEED)

## 2. Create Synthetic Prime-like Customer Data

**What**  
Create a small synthetic dataset representing member behavior.

**Why**  
The goal of this first lab is to learn the complete ML workflow rather than spend most of the session cleaning a real dataset.

**How**  
Simulate customer-level features such as tenure, purchase behavior, video engagement, delivery usage, support contacts, and recent activity change.

**Uniqueness / What to notice**  
These features are chosen because they have plausible connections to customer engagement and retention.  
Later we should be able to explain not only *whether* a model predicts churn, but *why*.

In [3]:
n = 5000

df = pd.DataFrame({
    "tenure_months": rng.integers(1, 120, n),
    "monthly_orders": rng.poisson(5, n),
    "video_hours_monthly": np.maximum(0, rng.normal(12, 8, n)),
    "prime_delivery_orders": rng.poisson(4, n),
    "avg_order_value": np.maximum(5, rng.normal(45, 20, n)),
    "support_contacts_90d": rng.poisson(1, n),
    "activity_drop_30d": rng.uniform(0, 1, n),
})

## 3. Define the Churn Mechanism

**What**  
Generate a churn probability and binary churn label.

**Why**  
For a supervised-learning problem, we need a target variable that has meaningful statistical relationships with the input features.

**How**  
Construct a latent log-odds score and convert it into probability with the sigmoid function.

**Uniqueness / What to notice**  
We intentionally encode directional relationships:

- higher recent activity drop → higher churn risk
- more support contacts → higher churn risk
- longer tenure → lower churn risk
- more orders / engagement → lower churn risk

Because we know the ground truth, we can later check whether our model recovers these relationships.

In [4]:
logit = (
    -1.5
    - 0.02 * df["tenure_months"]
    - 0.12 * df["monthly_orders"]
    - 0.03 * df["video_hours_monthly"]
    + 1.8 * df["activity_drop_30d"]
    + 0.35 * df["support_contacts_90d"]
)

prob_churn = 1 / (1 + np.exp(-logit))

df["churned"] = rng.binomial(1, prob_churn)

## 4. Basic Data Validation

**What**  
Inspect the dataset before modeling.

**Why**  
A model should never be trained before verifying that the dataset roughly matches our assumptions.

**How**  
Check sample rows, shape, missing values, and target class balance.

**Uniqueness / What to notice**  
Class balance matters because metrics such as accuracy can become misleading when one class dominates.

In [5]:
df.head()

,tenure_months,monthly_orders,video_hours_monthly,prime_delivery_orders,avg_order_value,support_contacts_90d,activity_drop_30d,churned
0,11,4,9.517125,3,30.529407,1,0.547990,0
1,93,6,15.715507,7,22.911671,1,0.755516,0
2,78,2,12.769933,0,35.831993,1,0.616509,1
3,53,4,8.900996,5,56.281166,0,0.811375,0
4,52,5,12.750004,5,50.397210,0,0.773981,0


In [6]:
df.shape

(5000, 8)

In [7]:
df.isna().sum()

,0
tenure_months,0
monthly_orders,0
video_hours_monthly,0
prime_delivery_orders,0
avg_order_value,0
support_contacts_90d,0
activity_drop_30d,0
churned,0


In [8]:
df["churned"].value_counts(normalize=True)

,proportion
churned,
0,0.8898
1,0.1102


## 5. Train / Validation / Test Split

**What**  
Divide the dataset into training, validation, and test sets.

**Why**  
Using the same data for both learning and evaluation can lead to overly optimistic performance estimates.

**How**

- Training set: fit model parameters
- Validation set: choose models, hyperparameters, and thresholds
- Test set: final evaluation after model decisions are complete

**Uniqueness / What to notice**  
`stratify=y` preserves approximately the same churn/non-churn ratio across the splits.

This is particularly useful for imbalanced classification problems.

In [9]:
X = df.drop(columns="churned")
y = df["churned"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)

In [10]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nChurn rates:")
print("Train:", y_train.mean())
print("Validation:", y_val.mean())
print("Test:", y_test.mean())

Train: (3500, 7)
Validation: (750, 7)
Test: (750, 7)

Churn rates:
Train: 0.11028571428571429
Validation: 0.11066666666666666
Test: 0.10933333333333334


## Next Session — Logistic Regression Baseline

**What**  
Train our first churn prediction model.

**Why**  
Logistic Regression provides a simple, fast, and interpretable baseline for binary classification.

**How**  
Fit the model on the training set and evaluate it using validation data.

**Uniqueness / What to notice**  
Because the synthetic target was generated from a logistic relationship, Logistic Regression should be a particularly informative baseline.

We will ask:

1. How well does the model identify churners?
2. Which features increase or decrease predicted churn risk?
3. Which evaluation metric best matches the business objective?
4. What mistakes does the model make?
5. What retention action could follow from the prediction?